# 7장 · 퍼플렉시티 — LLM 의 성적표 📊

웹 챕터 **"퍼플렉시티"** 와 짝을 이루는 Python 실습입니다.
웹 채점기에서 잡은 감(PPL = 평균 몇 갈래 고민)을, 여기서는 **기하평균 검산과
"학습할수록 성적이 오르는" 곡선**으로 굳힙니다.

준비물: `pip install -r requirements.txt`

> 🤖 **AI 튜터**: 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요. 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)

In [ ]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요 (웹앱과 같은 로컬 AI, 클릭해서 열기)
from tutor import chat

chat('ch7')

## 1. 비트 ↔ 갈래 환산표

PPL = 2^H. 지금까지 계산한 엔트로피들을 전부 "갈래 수"로 되돌려 봅니다.
**균등 k갈래면 PPL = k 그대로** 나오는지도 확인하세요.

In [ ]:
import numpy as np

def entropy(p):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    return float(np.sum(p * np.log2(1 / p)))

def perplexity_of(p):
    """분포 자체의 퍼플렉시티 = 2^H(p) — 실질 갈래 수."""
    return 2 ** entropy(p)

cases = {
    '공정 동전 (½, ½)         ': [1/2, 1/2],
    '주사위 (⅙ × 6)           ': [1/6] * 6,
    '90:10 치우친 동전        ': [0.9, 0.1],
    '날씨 (½, ¼, ⅛, ⅛)        ': [1/2, 1/4, 1/8, 1/8],
    '몰빵 (1, 0, 0, 0)        ': [1, 0, 0, 0],
    '어휘 1024개 완전 균등    ': [1/1024] * 1024,
}

print('상황                      | H (비트) | PPL = 2^H (갈래)')
print('-' * 58)
for name, p in cases.items():
    print(f'{name} | {entropy(p):7.3f} | {perplexity_of(p):10.3f}')

# 균등 k갈래의 PPL 은 정확히 k
for k in [2, 6, 1024]:
    assert abs(perplexity_of([1/k] * k) - k) < 1e-6
print('\n✅ 균등 k갈래의 PPL = k 그대로. 그래서 PPL 을 "실질 갈래 수"로 읽습니다.')

## 2. 문장 채점: 그리고 기하평균 검산

4장에서 손실을 계산했던 그 문장을 이번엔 **끝까지**(2^ 까지) 채점합니다.
그리고 심화 카드의 주장(PPL = 정답 확률 역수들의 **기하평균**)을 검산해요.

In [ ]:
import matplotlib.pyplot as plt

# 한글 폰트 설정 (안 하면 한글 제목/라벨이 네모로 깨짐)
# Windows: 'Malgun Gothic' / macOS: 'AppleGothic' / Linux: 'NanumGothic' 로 바꾸세요
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False  # 음수 부호(-) 깨짐 방지

tokens = ['나는', '아침', '에', '따뜻한', '국밥', '을', '먹었다']
q = np.array([0.20, 0.30, 0.95, 0.10, 0.02, 0.90, 0.60])   # 4장과 같은 문장

losses = -np.log2(q)
avg = losses.mean()
ppl = 2 ** avg

geo_mean = np.prod(1 / q) ** (1 / len(q))    # (1/q 들의 곱)^(1/N) = 기하평균

print(f'토큰별 손실  : {np.round(losses, 2)}')
print(f'평균 손실    : {avg:.4f} 비트')
print(f'PPL = 2^평균 : {ppl:.4f} 갈래')
print(f'기하평균 검산: {geo_mean:.4f} 갈래')
assert abs(ppl - geo_mean) < 1e-9, '기하평균과 다르다?!'
print('\n✅ PPL = (1/q 들의 기하평균). 곱셈으로 쌓이는 확률에 공정한 평균입니다.')

fig, ax = plt.subplots(figsize=(8, 3.8))
bars = ax.bar(tokens, 1 / q, color=['#f97316' if v > 8 else '#6366f1' for v in 1 / q], alpha=0.9)
ax.bar_label(bars, fmt='%.0f')
ax.axhline(ppl, color='#10b981', lw=2, linestyle='--', label=f'문장 PPL = {ppl:.1f}갈래 (기하평균)')
ax.set_ylabel('토큰별 갈래 수 1/q')
ax.set_title('토큰마다 몇 갈래에서 고민했나: 국밥(50갈래)이 성적을 지배')
ax.legend()
plt.show()

## 3. 학습할수록 성적이 오른다: PPL 하강 곡선

6장 훈련소를 다시 돌리되, 이번엔 성적을 **PPL 단위**로 지켜봅니다.
어휘 4개 완전 균등(PPL 4)에서 출발해 어디까지 내려갈까요?

In [ ]:
def softmax(z):
    e = np.exp(z - z.max())
    return e / e.sum()

def cross_entropy(p, q):
    p, q = np.asarray(p, dtype=float), np.asarray(q, dtype=float)
    mask = p > 0
    return float(np.sum(p[mask] * np.log2(1 / q[mask])))

p_data = np.array([1/2, 1/4, 1/8, 1/8])
floor_ppl = 2 ** entropy(p_data)             # 완벽한 모델의 PPL = 2^1.75 ≈ 3.36

z = np.zeros(4)                              # 균등(PPL 4)에서 출발
ppls = []
for step in range(60):
    q_model = softmax(z)
    ppls.append(2 ** cross_entropy(p_data, q_model))
    z = z - 0.3 * (q_model - p_data)         # 경사하강 (6장)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ppls, color='#6366f1', lw=2.5, label='모델의 PPL')
ax.axhline(floor_ppl, color='#ef4444', lw=2, linestyle='--',
           label=f'완벽한 모델의 PPL = 2^H(p) = {floor_ppl:.2f}갈래')
ax.axhline(4, color='#94a3b8', lw=1, linestyle=':', label='출발점: 균등 4갈래')
ax.set_xlabel('경사하강 걸음 수')
ax.set_ylabel('퍼플렉시티 (갈래)')
ax.set_title('학습 = 성적(PPL) 끌어내리기: 균등 4갈래에서 실질 3.36갈래까지')
ax.grid(True, color='#eef2f7')
ax.legend()
plt.show()

print(f'출발 PPL {ppls[0]:.3f} → 최종 PPL {ppls[-1]:.3f} (바닥 {floor_ppl:.3f})')
print('실제 LLM 사전학습도 같은 그림이에요. 어휘 수만 갈래에서 출발해 수십 갈래로 내려오는 여정입니다.')

## 4. nat 단위 주의: 밑만 맞추면 같은 성적

딥러닝 프레임워크의 loss 는 보통 **nat(ln)** 단위입니다.
비트 loss 에 2^, nat loss 에 e^. 어느 쪽으로 계산해도 PPL 은 같아야겠죠?

In [ ]:
loss_bits = -np.log2(q).mean()    # 비트 단위 loss (우리 방식)
loss_nats = -np.log(q).mean()     # nat 단위 loss (프레임워크 방식)

print(f'비트 단위 loss : {loss_bits:.4f} → PPL = 2^loss = {2 ** loss_bits:.4f}')
print(f'nat  단위 loss : {loss_nats:.4f} → PPL = e^loss = {np.exp(loss_nats):.4f}')
assert abs(2 ** loss_bits - np.exp(loss_nats)) < 1e-9
print('\n✅ 밑(2 vs e)만 짝지어 주면 같은 성적. 1장 bit/nat 카드의 실전 버전입니다.')

## 5. 직접 풀어보기 ✏️

문장 4개 토큰에서 모델이 정답에 준 확률이 (1/2, 1/8, 1/2, 1/8) 이었습니다.
이 문장의 **퍼플렉시티**는? (웹 챕터의 StepQuiz 와 같은 계산)

In [ ]:
# TODO: 아래 answer 를 여러분의 계산으로 바꾸고 실행하세요.
# 힌트: 토큰 손실 1,3,1,3비트 → 평균 → 2^  (또는 기하평균으로 한 번에!)

qs = np.array([1/2, 1/8, 1/2, 1/8])
answer = 2 ** (-np.log2(qs).mean())
print('내 답:', answer)
assert abs(answer - 4) < 1e-9, '다시 계산해 보세요 — 평균 손실 2비트의 2^'
print('✅ 잘했어요! 평균 2비트 = 평균 4갈래 고민 수준입니다.')

---
**오늘의 한 줄**: PPL = 2^손실 = "평균 몇 갈래의 균등 후보에서 고민하나".
낮을수록 좋고, 토크나이저·평가 텍스트가 같을 때만 비교한다.

다음 챕터는 생성의 다이얼인 **temperature 가 분포의 엔트로피를 돌리는 원리**를
소프트맥스와 함께 파헤칩니다. 4부 12장의 top-k·top-p 도 재해석해요. 🙂